In [ ]:
import requests
from typing import List, Dict

def health_check(services: List[str], timeout: int = 3) -> Dict[str, str]:
    """
    Checks health of given HTTP services.

    :param services: List of service URLs
    :param timeout: Timeout in seconds for each request
    :return: Dictionary with service URL and health status
    """
    status = {}

    for service in services:
        try:
            response = requests.get(service, timeout=timeout)
            if response.status_code == 200:
                status[service] = "HEALTHY"
            else:
                status[service] = f"UNHEALTHY (Status: {response.status_code})"
        except requests.exceptions.RequestException as e:
            status[service] = f"DOWN ({str(e)})"

    return status


In [ ]:
import requests
from typing import List, Dict

def health_check(services: List[str], timeout: int = 3) -> Dict[str, str]:
    """
    Checks health of given HTTP services.

    :param services: List of service URLs
    :param timeout: Timeout in seconds for each request
    :return: Dictionary with service URL and health status
    """
    status = {}

    for service in services:
        try:
            response = requests.get(service, timeout=timeout)
            if response.status_code == 200:
                status[service] = "HEALTHY"
            else:
                status[service] = f"UNHEALTHY (Status: {response.status_code})"
        except requests.exceptions.RequestException as e:
            status[service] = f"DOWN ({str(e)})"

    return status


In [ ]:
services = [
    "https://google.com",
    "http://localhost:8080/health",
    "http://invalid-service"
]

results = health_check(services)

for service, result in results.items():
    print(f"{service} → {result}")


In [ ]:
import asyncio
import aiohttp
import socket
from typing import List, Dict


async def check_service(
    session: aiohttp.ClientSession,
    service: str,
    timeout: int
) -> str:

    try:
        async with session.get(service, timeout=timeout) as response:
            if response.status == 200:
                return "UP"
            elif 400 <= response.status < 500:
                return f"HTTP_{response.status}"
            elif 500 <= response.status < 600:
                return "HTTP_5XX"
            else:
                return f"HTTP_{response.status}"

    except asyncio.TimeoutError:
        return "TIMEOUT"

    except aiohttp.ClientConnectorError as e:
        if isinstance(e.os_error, socket.gaierror):
            return "DNS_FAILURE"
        elif isinstance(e.os_error, ConnectionRefusedError):
            return "CONNECTION_REFUSED"
        else:
            return "CONNECTION_ERROR"

    except aiohttp.ClientError:
        return "CLIENT_ERROR"

    except Exception:
        return "UNKNOWN_ERROR"


async def async_health_check(services: List[str],timeout: int = 3) -> Dict[str, str]:

    async with aiohttp.ClientSession() as session:
        tasks = {
            service: asyncio.create_task(
                check_service(session, service, timeout)
            )
            for service in services
        }

        return {service: await task for service, task in tasks.items()}


In [ ]:
if __name__ == "__main__":
    services = [
        "https://google.com",
        "http://localhost:8080/health",
        "http://invalid-service"
    ]
 #   results = asyncio.run(async_health_check(services))
    results = await async_health_check(services)
    for service, status in results.items():
        print(f"{service} → {status}")
